# TileDB Patch Array — Setup Notebook

## Schema Description

This notebook creates and seeds the TileDB array used for the
**Batch read of patch data for model training (TileDB)** benchmark.

### Array Design

| Component    | Value                                                           |
| ------------ | --------------------------------------------------------------- |
| Array type   | Dense (4-D)                                                     |
| Dimensions   | `patch_id` (int64, 0–999999), `h` (0–31), `w` (0–31), `c` (0–2) |
| Tile extents | patch_id=1024, h=32, w=32, c=3                                  |
| Attribute    | `pixel` — uint8, LZ4-compressed, fixed-size                     |
| Cell order   | row-major                                                       |
| Tile order   | row-major                                                       |

### Rationale

- **4-D dense layout** natively represents `(N, H, W, C)` patch tensors,
  matching the shape expected by DL frameworks.
- **Tile size 1024 along patch_id** balances chunk-read overhead vs.
  read-amplification for random access.
- **LZ4 compression** gives fast decompression at moderate compression
  ratios, suitable for uint8 image data.
- **patch_id** matches the primary-key `id` column of the relational patch
  table (see `db_technical_design.md`), enabling O(1) lookup when used
  together with `multi_index`.

### Setup Steps

1. Drop/recreate the TileDB array (idempotent).
2. Seed 1 000 000 patches (32×32×3, uint8) in batches of 10 000.
3. Provide a teardown cell that removes the array after benchmarking.


In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
import os
import time

import numpy as np
import tiledb

print(f"TileDB version : {tiledb.__version__}")
print(f"NumPy  version : {np.__version__}")

# ── Config ────────────────────────────────────────────────────────────────────
TILEDB_PATH   = "/tmp/tiledb_patch_benchmark_1M"
NUM_PATCHES   = 1_000_000
PATCH_H       = 32
PATCH_W       = 32
PATCH_C       = 3
SEED_BATCH    = 10_000       # write N patches per fragment to cap memory use
TILE_PATCH_ID = 1024         # tile extent along patch_id dimension

print(f"\nArray path     : {TILEDB_PATH}")
print(f"Total patches  : {NUM_PATCHES:,}")
print(f"Patch shape    : ({PATCH_H}, {PATCH_W}, {PATCH_C})")
estimated_gb = NUM_PATCHES * PATCH_H * PATCH_W * PATCH_C / 1e9
print(f"Uncompressed   : ~{estimated_gb:.1f} GB")

In [ ]:
# ── Idempotent array creation ─────────────────────────────────────────────────
if tiledb.array_exists(TILEDB_PATH):
    print(f"Removing existing array: {TILEDB_PATH}")
    tiledb.remove(TILEDB_PATH)

domain = tiledb.Domain(
    tiledb.Dim(
        name="patch_id",
        domain=(0, NUM_PATCHES - 1),
        tile=TILE_PATCH_ID,
        dtype=np.int64,
    ),
    tiledb.Dim(name="h", domain=(0, PATCH_H - 1), tile=PATCH_H, dtype=np.int64),
    tiledb.Dim(name="w", domain=(0, PATCH_W - 1), tile=PATCH_W, dtype=np.int64),
    tiledb.Dim(name="c", domain=(0, PATCH_C - 1), tile=PATCH_C, dtype=np.int64),
)

pixel_attr = tiledb.Attr(
    name="pixel",
    dtype=np.uint8,
    var=False,
    filters=tiledb.FilterList([tiledb.LZ4Filter()]),
)

schema = tiledb.ArraySchema(
    domain=domain,
    attrs=[pixel_attr],
    sparse=False,
    cell_order="row-major",
    tile_order="row-major",
)

tiledb.DenseArray.create(TILEDB_PATH, schema)
print(f"Created TileDB dense array at: {TILEDB_PATH}")
print(schema)

In [ ]:
# ── Seed data ─────────────────────────────────────────────────────────────────
# Write in batches of SEED_BATCH to keep peak RAM usage under ~1 GB
rng = np.random.default_rng(2024)

t_seed_start = time.perf_counter()
print("Seeding data…")

for batch_start in range(0, NUM_PATCHES, SEED_BATCH):
    batch_end = min(batch_start + SEED_BATCH, NUM_PATCHES)
    chunk = rng.integers(
        0, 256,
        size=(batch_end - batch_start, PATCH_H, PATCH_W, PATCH_C),
        dtype=np.uint8,
    )
    with tiledb.DenseArray(TILEDB_PATH, mode="w") as A:
        A[batch_start:batch_end] = {"pixel": chunk}

    if batch_start % 100_000 == 0:
        elapsed = time.perf_counter() - t_seed_start
        print(f"  {batch_start:>9,} / {NUM_PATCHES:,}  ({elapsed:.1f}s elapsed)")

t_seed_total = time.perf_counter() - t_seed_start
print(f"Seeding complete — {t_seed_total:.1f}s total")

In [ ]:
# ── Verify array integrity ────────────────────────────────────────────────────
with tiledb.DenseArray(TILEDB_PATH, mode="r") as A:
    sample = A[0:5]["pixel"]
    print(f"Sample read shape  : {sample.shape}")
    print(f"Sample dtype       : {sample.dtype}")
    print(f"Sample value range : [{sample.min()}, {sample.max()}]")
print("Array is ready for benchmarking.")

In [ ]:
# ── Teardown (run after benchmarking) ─────────────────────────────────────────
# Uncomment to remove the array once benchmarking is complete.
# if tiledb.array_exists(TILEDB_PATH):
#     tiledb.remove(TILEDB_PATH)
#     print(f"Removed TileDB array: {TILEDB_PATH}")